In [20]:
import numpy as np
tour = [1, 2, 3, 4, 5]

In [21]:
tour_shifted = np.roll(tour, -1)
tour_shifted

array([2, 3, 4, 5, 1])

In [22]:
def tour_distance(tour, dist_matrix):
    print(tour)
    tour_shifted = np.roll(tour, -1)
    print(tour_shifted)
    return np.sum(np.asarray(dist_matrix)[tour, tour_shifted])

In [23]:
import numpy as np

tour = np.array([0, 1, 2, 3])  # visits 0→1→2→3→0
dist_matrix = np.array([
    [0, 10, 20, 30],
    [10, 0, 15, 25],
    [20, 15, 0, 18],
    [30, 25, 18, 0],
])

total = tour_distance(tour, dist_matrix)
print(total)  # 10 + 15 + 18 + 30 = 73

[0 1 2 3]
[1 2 3 0]
73


## Testing 2-opt

In [3]:
tour

[1, 2, 3, 4, 5]

In [1]:
def two_opt(tour, i, j):
    new_tour = np.concatenate((tour[:i], tour[i:j+1][::-1], tour[j+1:]))
    return new_tour

In [5]:
def _two_opt_first_improvement(tour: np.ndarray) -> tuple[np.ndarray, bool]:
    current_distance = 10
    length = len(tour) - 1 if tour[0] == tour[-1] else len(tour)
    for i in range(1, length - 1):
        for j in range(i + 1, length):
            if j - i == 1:
                continue
            print(tour, i, j)
            candidate = two_opt(tour, i, j)
            print("New tour:", candidate)
            candidate_distance = 11
            if candidate_distance < current_distance:
                return candidate, True
    return tour, False

In [6]:
_two_opt_first_improvement(tour)

[1, 2, 3, 4, 5] 1 3
New tour: [1 4 3 2 5]
[1, 2, 3, 4, 5] 1 4
New tour: [1. 5. 4. 3. 2.]
[1, 2, 3, 4, 5] 2 4
New tour: [1. 2. 5. 4. 3.]


([1, 2, 3, 4, 5], False)

In [7]:
tour = [1, 2, 3, 4, 5, 6, 7]

In [11]:
def double_bridge_move(tour: np.ndarray, a: int, b: int, c: int, d: int) -> np.ndarray:
    """Apply a double-bridge move defined by four cut indices."""
    if len(tour) < 6:
        return tour.copy()

    is_closed = tour[0] == tour[-1]
    core = tour[:-1] if is_closed else tour
    n = len(core)
    print("Core:", core)
    if n < 6:
        return tour.copy()

    segment_1 = core[:a]
    segment_2 = core[a:b]
    segment_3 = core[b:c]
    segment_4 = core[c:d]
    segment_5 = core[d:]
    print("Segments:", segment_1, segment_2, segment_3, segment_4, segment_5)

    new_core = np.concatenate((segment_1, segment_3, segment_2, segment_4, segment_5))
    if is_closed:
        new_core = np.concatenate((new_core, [new_core[0]]))
    return new_core

In [12]:
def _double_bridge_first_improvement(tour: np.ndarray) -> tuple[np.ndarray, bool]:
    current_distance = 10
    is_closed = tour[0] == tour[-1]
    length = len(tour) - 1 if is_closed else len(tour)
    if length < 6:
        return tour, False

    for a in range(1, length - 3):
        for b in range(a + 1, length - 2):
            for c in range(b + 1, length - 1):
                for d in range(c + 1, length):
                    print("Current tour:", tour, "Cuts:", a, b, c, d)
                    candidate = double_bridge_move(tour, a, b, c, d)
                    print("New tour:", candidate)
                    candidate_distance = 11
                    if candidate_distance < current_distance:
                        return candidate, True
    return tour, False

In [13]:
_double_bridge_first_improvement(tour)

Current tour: [1, 2, 3, 4, 5, 6, 7] Cuts: 1 2 3 4
Core: [1, 2, 3, 4, 5, 6, 7]
Segments: [1] [2] [3] [4] [5, 6, 7]
New tour: [1 3 2 4 5 6 7]
Current tour: [1, 2, 3, 4, 5, 6, 7] Cuts: 1 2 3 5
Core: [1, 2, 3, 4, 5, 6, 7]
Segments: [1] [2] [3] [4, 5] [6, 7]
New tour: [1 3 2 4 5 6 7]
Current tour: [1, 2, 3, 4, 5, 6, 7] Cuts: 1 2 3 6
Core: [1, 2, 3, 4, 5, 6, 7]
Segments: [1] [2] [3] [4, 5, 6] [7]
New tour: [1 3 2 4 5 6 7]
Current tour: [1, 2, 3, 4, 5, 6, 7] Cuts: 1 2 4 5
Core: [1, 2, 3, 4, 5, 6, 7]
Segments: [1] [2] [3, 4] [5] [6, 7]
New tour: [1 3 4 2 5 6 7]
Current tour: [1, 2, 3, 4, 5, 6, 7] Cuts: 1 2 4 6
Core: [1, 2, 3, 4, 5, 6, 7]
Segments: [1] [2] [3, 4] [5, 6] [7]
New tour: [1 3 4 2 5 6 7]
Current tour: [1, 2, 3, 4, 5, 6, 7] Cuts: 1 2 5 6
Core: [1, 2, 3, 4, 5, 6, 7]
Segments: [1] [2] [3, 4, 5] [6] [7]
New tour: [1 3 4 5 2 6 7]
Current tour: [1, 2, 3, 4, 5, 6, 7] Cuts: 1 3 4 5
Core: [1, 2, 3, 4, 5, 6, 7]
Segments: [1] [2, 3] [4] [5] [6, 7]
New tour: [1 4 2 3 5 6 7]
Current tour: [1, 2,

([1, 2, 3, 4, 5, 6, 7], False)

In [ ]:
def _reward_matrix(distance_matrix: np.ndarray) -> np.ndarray:
	"""Build the reward matrix r(s,a) = M_i / d_ij as described in the paper."""
	with np.errstate(divide="ignore"):
		mean_dist = distance_matrix.mean(axis=1)
		print(mean_dist)
		rewards = np.divide(mean_dist[:, None], distance_matrix, where=distance_matrix > 0)
	rewards[np.isinf(rewards)] = 0.0
	rewards[np.isnan(rewards)] = 0.0
	return rewards




In [9]:
import numpy as np

sample_dist = np.array([
    [0.0, 2.0, 4.0],
    [2.0, 0.0, 6.0],
    [4.0, 6.0, 0.0],
])

sample_rewards = _reward_matrix(sample_dist)
sample_rewards

[2.         2.66666667 3.33333333]
[[2.        ]
 [2.66666667]
 [3.33333333]]


array([[0.        , 1.        , 0.5       ],
       [1.33333333, 0.        , 0.44444444],
       [0.83333333, 0.55555556, 0.        ]])

### Testing q-learning initial solution

In [3]:
import sys
from pathlib import Path

repo_root = Path.cwd().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [4]:
src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
src_dir

PosixPath('/Users/karinaassiniandreatta/Documents/01 phd/codes/tsp_rl_metaheuristic/src')

In [27]:
from pathlib import Path
import numpy as np

from src.structures.graph import Graph
from src.solver.initial_solution import q_learning_tour, QLearningConfig, nearest_neighbour_tour

instance_path = Path("/Users/karinaassiniandreatta/Documents/01 phd/codes/tsp_rl_metaheuristic/instances/tsplib/swiss42.tsp")
graph = Graph.from_tsplib(instance_path)


In [ ]:
def tour_distance(tour, dist_matrix):
    tour_shifted = np.roll(tour, -1)
    return np.sum(dist_matrix[tour, tour_shifted])

In [ ]:

cfg = QLearningConfig(
    episodes=1000,
    alpha=0.3,
    gamma=0.9,
    epsilon=0.4,
    epsilon_decay=0.995,
    epsilon_min=0.05,
    #cache_dir=Path("/Users/karinaassiniandreatta/Documents/01 phd/codes/tsp_rl_metaheuristic/data/q_learning_cache"),
)
tour_q = q_learning_tour(graph, start=0, cfg=cfg)
tour_q, len(tour_q)

(array([ 0, 27,  2, 30, 16, 14, 15, 12, 11, 20, 33, 34, 26,  5,  3, 31, 17,
        23, 41, 10, 25, 21,  9, 19,  6,  1, 28, 29, 37, 36, 35, 22, 39, 18,
        40, 24,  4, 13,  8,  7, 32, 38,  0]),
 43)

In [37]:
distance_matrix = np.asarray(graph.get_distance_matrix(), dtype=float)

In [42]:
tour_distance(tour_q, distance_matrix)

np.float64(3327.0)

In [43]:
tour = nearest_neighbour_tour(graph)

In [44]:
tour_distance(tour, distance_matrix)

np.float64(1564.0)